In [0]:
pip install langchain

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#Enable mlflow tracing for LangChain
import mlflow
mlflow.langchain.autolog()

In [0]:
%pip install databricks-langchain

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install opentelemetry-proto
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Create a langchain agent
from langchain.agents import create_agent 
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool 
from langgraph.checkpoint.memory import InMemorySaver

def build_agent(lIm_endpoint: str, index_name: str, num_results: int = 3):
    model = ChatDatabricks (
        endpoint=lIm_endpoint, 
        max_tokens=500,
    )
    vs_tool = VectorSearchRetrieverTool(
        name="invoice_knowledge_search", 
        index_name=index_name,
        description="Search invoice, Receipts knowledge base for relevant information", 
        num_results=num_results,
    )

    # Optional: use a in memory saver to save the agent's state
    checkpointer = InMemorySaver()

    system_prompt = """You are the Invoice Knowledge Assistant (IKA). Respond in a clear, professional, and factual tone
    appropriate for user queries. Use only verified information from internal documents, and include source references when available. If the answer cannot be found, clearly state that and suggest related sections or next steps. Do not speculate, make assumptions, or provide information outside the provided context."""

    agent = create_agent(
        model= model, 
        tools= [vs_tool],
        system_prompt= system_prompt, 
        checkpointer= checkpointer,
    )
    return agent

# 'thread_id' is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "databricks-demo-4"}}
vs_index_name = "idp.processed.docs_chunked_vs_index"

# Quick smoke test
llm_endpoint_name = "databricks-meta-llama-3-3-70b-instruct"
agent = build_agent(llm_endpoint_name, vs_index_name, 3)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Who has high total in invoices?"}]}, 
    config=config
)
print(response['messages'][-1]. content)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Based on the function output, the invoice with the high total is $5,219.45, which is associated with the document having an id of 86.0.


Trace(trace_id=tr-8b927c35a3294cbd9a52bf929db60c81)